In [ ]:
#| default_exp cam_pub

In [ ]:
#| hide
from nbdev.showdoc import *

# Standard Publisher

> A well-structured, reusable ROS2 camera publisher node. Refactors the quick-and-dirty prototype from `01_simple_example` into production-quality code.

## Overview

In the previous notebook (`01_simple_example`) we built a basic `CameraNode` that captures frames and publishes them. That node works, but it's a prototype — hard to extend, hard to test, and tightly coupled to OpenCV.

In this notebook we refactor the publisher into a clean, reusable node:

- Separated camera logic from ROS2 publishing logic
- Clear parameter handling with validation
- Proper resource management
- Ready to be extended in later notebooks (parameters, services, lifecycle)

## Imports

In [ ]:
#| export
#| hide
import sys
import rclpy
from rclpy.node import Node
from rclpy.executors import ExternalShutdownException
import cv2
from cv_bridge import CvBridge
from sensor_msgs.msg import Image

## Camera Wrapper

Instead of putting all camera logic inside the ROS2 node, we extract it into a simple `Camera` class. This makes the code testable and reusable.

In [ ]:
#| export
class Camera:
    """Wrapper around `cv2.VideoCapture` with format and resolution helpers."""
    def __init__(self, source='0', fourcc='MJPG', width=640, height=480):
        self.source = source
        if source.isdigit():
            self.cap = cv2.VideoCapture(int(source))
        else:
            self.cap = cv2.VideoCapture(source)

        if not self.cap.isOpened():
            raise RuntimeError(f'Could not open video source: {source}')

        self.set_fourcc(fourcc)
        self.set_resolution(width, height)

    def set_fourcc(self, fmt):
        """Set pixel/compression format (FOURCC). e.g. 'MJPG', 'YUYV'."""
        self.cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*fmt))

    def get_fourcc(self):
        """Get current FOURCC format as string."""
        codec_int = int(self.cap.get(cv2.CAP_PROP_FOURCC))
        return ''.join([chr((codec_int >> 8 * i) & 0xFF) for i in range(4)])

    def set_resolution(self, width, height):
        """Set capture resolution."""
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, width)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, height)

    def get_resolution(self):
        """Get current capture resolution as (width, height)."""
        w = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        return w, h

    def read(self):
        """Capture a frame. Returns (success, frame)."""
        return self.cap.read()

    def release(self):
        """Release the camera resource."""
        self.cap.release()

## CamPublisher Node

`CamPublisher` is a ROS2 node that uses the `Camera` wrapper to capture frames and publish them as `sensor_msgs/Image` messages.

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `video_source` | string | `'0'` | Camera index or stream URL |
| `frame_rate` | float | `30.0` | Publishing frequency in FPS |
| `topic_name` | string | `'/camera/image_raw'` | Topic to publish images on |
| `video_fourcc` | string | `'MJPG'` | Compression format |
| `resolution_width` | int | `640` | Output frame width |
| `resolution_height` | int | `480` | Output frame height |

In [ ]:
#| export
class CamPublisher(Node):
    """ROS2 node that publishes camera frames as `sensor_msgs/Image`."""
    def __init__(self):
        super().__init__('cam_pub')

        # Declare parameters
        self.declare_parameter('video_source', '0')
        self.declare_parameter('frame_rate', 30.0)
        self.declare_parameter('topic_name', '/camera/image_raw')
        self.declare_parameter('video_fourcc', 'MJPG')
        self.declare_parameter('resolution_width', 640)
        self.declare_parameter('resolution_height', 480)

        # Read parameters
        source = self.get_parameter('video_source').value
        self.frame_rate = self.get_parameter('frame_rate').value
        self.topic_name = self.get_parameter('topic_name').value
        fourcc = self.get_parameter('video_fourcc').value
        width = self.get_parameter('resolution_width').value
        height = self.get_parameter('resolution_height').value

        # Initialize camera
        self.camera = Camera(source, fourcc, width, height)

        # Publisher
        self.publisher_ = self.create_publisher(Image, self.topic_name, 20)

        # Timer
        period = 1.0 / self.frame_rate
        self.timer = self.create_timer(period, self._publish_frame)

        # CvBridge
        self.bridge = CvBridge()
        self.frame_count = 0

        self.get_logger().info(
            f'Publishing at {self.frame_rate} FPS from {source} on "{self.topic_name}"'
        )

    def _publish_frame(self):
        """Capture a frame and publish it."""
        ret, frame = self.camera.read()
        if not ret:
            self.get_logger().warn('Failed to read frame')
            return

        msg = self.bridge.cv2_to_imgmsg(frame, encoding='bgr8')
        self.publisher_.publish(msg)

        if self.frame_count % 10 == 0:
            self.get_logger().info(f'Published frame {self.frame_count}')
        self.frame_count += 1

    def destroy_node(self):
        """Release the camera before destroying the node."""
        self.camera.release()
        super().destroy_node()

### show_doc

In [ ]:
show_doc(CamPublisher)

## Entry Point

Every ROS2 node must define `main(args=None)` so `ros2 run` can discover and launch it.

In [ ]:
#| export
def main(args=None):
    """Initialize ROS2, spin the CamPublisher, and handle shutdown."""
    rclpy.init(args=args)
    node = CamPublisher()

    try:
        rclpy.spin(node)
    except (KeyboardInterrupt, ExternalShutdownException):
        node.get_logger().info('Node stopped by user or external shutdown')
    finally:
        node.destroy_node()
        rclpy.try_shutdown()

In [ ]:
#| exporti
if not hasattr(sys, 'ps1'):
    if __name__ == "__main__":
        main()

> The `sys.ps1` check prevents the node from launching when the module is imported in an interactive session (notebook or REPL). This way, `python3 cam_pub.py` starts the node, but `import cam_pub` in a notebook does not.

## Usage

First, export the notebook to generate the Python module:

```bash
uv run nbdev-export
```

This creates `ros2_cam_nbdev/cam_pub.py`.

### Method 1 — Run directly with Python

The fastest way to test. No workspace or compilation needed.

**Terminal 1** — run the node:

```bash
uv run python -m ros2_cam_nbdev.cam_pub
```

**Terminal 2** — check the topic is publishing:

```bash
ros2 topic hz /camera/image_raw
```

See `08_visualization` for tools to view the images.

You can override parameters at launch:

```bash
uv run python -m ros2_cam_nbdev.cam_pub --ros-args \
    -p video_source:="http://192.168.55.178:8080/video" \
    -p frame_rate:=15.0 \
    -p resolution_width:=1280 \
    -p resolution_height:=720
```

### Method 2 — Full ROS2 workflow

The standard ROS2 way: create a workspace, package, compile, and run.

**Step 1** — Source ROS2:

```bash
source /opt/ros/$ROS_DISTRO/setup.bash
```

> Use `setup.zsh` if your shell is zsh.

**Step 2** — Create the workspace:

```bash
zsh scripts/generate_ros2_ws.zsh new_ros2_ws
```

**Step 3** — Create the package:

```bash
zsh scripts/mk_ros2_pkg.sh new_ros2_ws my_camera_pkg camera_opencv.txt
```

> This creates `new_ros2_ws/src/my_camera_pkg/` with ROS2 dependencies
> read from `scripts/pkg_dependencies/camera_opencv.txt`.

**Step 4** — Copy the node into the package:

```bash
cp ros2_cam_nbdev/cam_pub.py new_ros2_ws/src/my_camera_pkg/my_camera_pkg/
```

**Step 5** — Compile:

```bash
bash scripts/compile_pkg.sh new_ros2_ws my_camera_pkg
```

> `compile_pkg.sh` generates `setup.py` automatically (entry points are
> discovered from any file with `def main(args=None):`) and runs `colcon build`.

**Step 6** — Source the workspace overlay:

```bash
source new_ros2_ws/install/setup.bash
```

> Use `setup.zsh` if your shell is zsh.

**Step 7** — Run the node:

```bash
ros2 run my_camera_pkg cam_pub
```

**Terminal 2** — verify and visualize:

```bash
ros2 topic hz /camera/image_raw
```

See `08_visualization` for tools to view the images.

## Visualize

With the node running, open a second terminal and visualize the images:

```sh
ros2 run rqt_image_view rqt_image_view
```

Select `/camera/image_raw` from the dropdown.

For more visualization options (rviz2, Foxglove), see `08_visualization`.

## Next Steps

This node works, but every parameter must be set at launch time. In the next notebook (`03_parameters_and_services`) we add runtime parameter updates and services for capture and reset.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()